# Step 3: ESM-2 Protein Embeddings
## Comparative Lassa–Ebola Mutation Study

This notebook:
1. Installs all required dependencies
2. Downloads cleaned FASTA sequences from GitHub
3. Loads the ESM-2 650M parameter model
4. Generates 1,280-dimensional embeddings for 2,390 sequences
5. Performs PCA analysis and visualisation
6. Saves all embeddings and metadata
7. Produces quality metrics and publication-ready plots
8. Exports a full summary report

## Section 1 – Setup & Dependencies

In [ ]:
import subprocess, sys

def pip_install(*packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])

pip_install('torch', 'torchvision')
pip_install('fair-esm')
pip_install('biopython')
pip_install('numpy', 'pandas', 'matplotlib', 'seaborn', 'scikit-learn', 'tqdm')

print('All dependencies installed successfully.')

In [ ]:
import os
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import esm
from Bio import SeqIO
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Device
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')
print(f'PyTorch version: {torch.__version__}')
print(f'ESM version: {esm.__version__}')

## Section 2 – Configuration

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
BASE_URL = (
    'https://raw.githubusercontent.com/'
    'Damilola-max/Comparative_Lassa_Ebola-Model/main/data/cleaned/'
)
LASSA_URL  = BASE_URL + 'lassa_cleaned.fasta'
EBOLA_URL  = BASE_URL + 'ebola_cleaned.fasta'

DATA_DIR   = 'data/cleaned'
EMBED_DIR  = 'data/embeddings'
FIG_DIR    = 'results/figures'

os.makedirs(DATA_DIR,  exist_ok=True)
os.makedirs(EMBED_DIR, exist_ok=True)
os.makedirs(FIG_DIR,   exist_ok=True)

LASSA_FASTA = os.path.join(DATA_DIR, 'lassa_cleaned.fasta')
EBOLA_FASTA = os.path.join(DATA_DIR, 'ebola_cleaned.fasta')

# ── ESM-2 model ──────────────────────────────────────────────────────────────
# esm2_t33_650M_UR50D  →  33 layers, 650 M params, 1280-D embeddings
ESM_MODEL_NAME = 'esm2_t33_650M_UR50D'
REPR_LAYER     = 33          # last transformer layer
EMBED_DIM      = 1280

# ── Batching ─────────────────────────────────────────────────────────────────
# Reduce batch size if you run out of memory; increase for speed on GPU.
BATCH_SIZE = 4 if DEVICE == 'cuda' else 1
# Maximum sequence length (tokens) per batch element before truncation
MAX_LEN = 1022   # ESM-2 hard limit is 1024 minus two special tokens

print('Configuration set.')

## Section 3 – Download Cleaned Sequences from GitHub

In [ ]:
def download_if_missing(url: str, dest: str) -> None:
    if os.path.exists(dest):
        print(f'  Already present: {dest}')
        return
    print(f'  Downloading {url} ...')
    urllib.request.urlretrieve(url, dest)
    print(f'  Saved to {dest}')

print('Downloading FASTA files …')
download_if_missing(LASSA_URL,  LASSA_FASTA)
download_if_missing(EBOLA_URL,  EBOLA_FASTA)
print('Done.')

## Section 4 – Load & Validate Sequences

In [ ]:
def load_fasta(path: str, label: str):
    records = []
    for rec in SeqIO.parse(path, 'fasta'):
        seq = str(rec.seq).upper().replace('*', '').replace('-', '')
        if len(seq) < 10:
            continue
        records.append({
            'id':    rec.id,
            'label': label,
            'seq':   seq,
            'length': len(seq),
        })
    return records

lassa_records = load_fasta(LASSA_FASTA, 'Lassa')
ebola_records = load_fasta(EBOLA_FASTA, 'Ebola')
all_records   = lassa_records + ebola_records

print(f'Lassa sequences: {len(lassa_records)}')
print(f'Ebola sequences: {len(ebola_records)}')
print(f'Total:           {len(all_records)}')

# Quick length stats
lengths = [r['length'] for r in all_records]
print(f'Length – min: {min(lengths)},  max: {max(lengths)},  '
      f'mean: {np.mean(lengths):.0f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, records, colour, title in zip(
    axes,
    [lassa_records, ebola_records],
    ['steelblue', 'tomato'],
    ['Lassa Protein Lengths', 'Ebola Protein Lengths'],
):
    lens = [r['length'] for r in records]
    ax.hist(lens, bins=40, color=colour, edgecolor='white', alpha=0.85)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Sequence length (aa)', fontsize=11)
    ax.set_ylabel('Count', fontsize=11)
    ax.axvline(np.mean(lens), color='black', linestyle='--',
               label=f'Mean: {np.mean(lens):.0f} aa')
    ax.legend(fontsize=10)

plt.suptitle('Sequence Length Distributions', fontsize=15, fontweight='bold')
plt.tight_layout()
out_path = os.path.join(FIG_DIR, '03_01_length_distribution.png')
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')

## Section 5 – Load ESM-2 Model (650 M parameters)

We use Meta's **ESM-2** model `esm2_t33_650M_UR50D` which produces
**1,280-dimensional** mean-pooled embeddings per sequence.

In [ ]:
print(f'Loading {ESM_MODEL_NAME} …')
model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
batch_converter = alphabet.get_batch_converter()
model = model.to(DEVICE)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f'Model loaded.  Parameters: {n_params:,}')
print(f'Embedding dimension: {EMBED_DIM}')

## Section 6 – Generate Embeddings

Sequences are processed in small batches.  
Long sequences are **truncated to 1,022 residues** (ESM-2 hard limit).  
Each embedding is the **mean of all per-residue representations**.

In [ ]:
@torch.no_grad()
def embed_batch(batch_data):
    """
    batch_data: list of (label, sequence) tuples
    Returns: numpy array of shape (len(batch_data), EMBED_DIM)
    """
    # Truncate sequences to MAX_LEN
    batch_data = [(lbl, seq[:MAX_LEN]) for lbl, seq in batch_data]
    batch_labels, batch_strs, batch_tokens = batch_converter(batch_data)
    batch_tokens = batch_tokens.to(DEVICE)

    results = model(batch_tokens, repr_layers=[REPR_LAYER],
                    return_contacts=False)
    token_representations = results['representations'][REPR_LAYER]

    embeddings = []
    for i, (_, seq) in enumerate(batch_data):
        # tokens: [BOS, aa1, aa2, …, EOS]  →  slice [1 : len(seq)+1]
        emb = token_representations[i, 1:len(seq) + 1].mean(0)
        embeddings.append(emb.cpu().numpy())

    return np.array(embeddings, dtype=np.float32)

print('Embedding function ready.')

In [ ]:
all_embeddings = []
failed_ids     = []

batch_input = [(r['id'], r['seq']) for r in all_records]
n_seq = len(batch_input)

print(f'Generating embeddings for {n_seq} sequences …')
for start in tqdm(range(0, n_seq, BATCH_SIZE), desc='Embedding batches'):
    batch = batch_input[start : start + BATCH_SIZE]
    try:
        embs = embed_batch(batch)
        all_embeddings.append(embs)
    except Exception as exc:
        print(f'  Batch {start}–{start+len(batch)} failed: {exc}')
        failed_ids.extend([b[0] for b in batch])
        # Append zero vectors so indices stay aligned
        all_embeddings.append(np.zeros((len(batch), EMBED_DIM), dtype=np.float32))

    # Free GPU cache periodically
    if DEVICE == 'cuda' and (start // BATCH_SIZE) % 50 == 0:
        torch.cuda.empty_cache()

embedding_matrix = np.vstack(all_embeddings)   # (N, 1280)
labels_array     = np.array([r['label']  for r in all_records])
ids_array        = np.array([r['id']     for r in all_records])
lengths_array    = np.array([r['length'] for r in all_records])

print(f'Embedding matrix shape: {embedding_matrix.shape}')
print(f'Failed sequences: {len(failed_ids)}')

## Section 7 – Save Embeddings & Metadata

In [ ]:
# ── Compressed NPZ (all sequences) ──────────────────────────────────────────
npz_path = os.path.join(EMBED_DIR, 'all_embeddings.npz')
np.savez_compressed(
    npz_path,
    embeddings = embedding_matrix,
    labels     = labels_array,
    ids        = ids_array,
    lengths    = lengths_array,
)
print(f'Saved compressed embeddings: {npz_path}')

# ── Per-virus NPY files ───────────────────────────────────────────────────────
lassa_mask = labels_array == 'Lassa'
ebola_mask = labels_array == 'Ebola'

lassa_npy = os.path.join(EMBED_DIR, 'lassa_embeddings.npy')
ebola_npy = os.path.join(EMBED_DIR, 'ebola_embeddings.npy')
np.save(lassa_npy, embedding_matrix[lassa_mask])
np.save(ebola_npy, embedding_matrix[ebola_mask])
print(f'Lassa embeddings: {embedding_matrix[lassa_mask].shape}  → {lassa_npy}')
print(f'Ebola embeddings: {embedding_matrix[ebola_mask].shape}  → {ebola_npy}')

# ── Metadata CSV ─────────────────────────────────────────────────────────────
meta_df = pd.DataFrame({
    'id':     ids_array,
    'label':  labels_array,
    'length': lengths_array,
})
meta_path = os.path.join(EMBED_DIR, 'embedding_metadata.csv')
meta_df.to_csv(meta_path, index=False)
print(f'Metadata saved: {meta_path}')

## Section 8 – PCA Analysis

In [ ]:
# Standardise before PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(embedding_matrix)

# PCA – keep enough components to explain ≥90% variance or at least 50
pca_full = PCA(n_components=min(50, embedding_matrix.shape[0],
                                embedding_matrix.shape[1]))
X_pca_full = pca_full.fit_transform(X_scaled)

# 2-D projection for visualisation
pca_2d = PCA(n_components=2, random_state=SEED)
X_2d   = pca_2d.fit_transform(X_scaled)

var_exp = pca_full.explained_variance_ratio_
cum_var = np.cumsum(var_exp)
n90 = int(np.searchsorted(cum_var, 0.90)) + 1

print(f'PC1 explains {var_exp[0]*100:.1f}% of variance')
print(f'PC2 explains {var_exp[1]*100:.1f}% of variance')
print(f'First {n90} PCs explain ≥90% of variance')

In [ ]:
colours = {'Lassa': '#2196F3', 'Ebola': '#F44336'}

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# ── Left: 2-D scatter ───────────────────────────────────────────────────────
ax = axes[0]
for label, colour in colours.items():
    mask = labels_array == label
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
               c=colour, label=label, alpha=0.5, s=10, linewidths=0)
ax.set_xlabel(f'PC1 ({var_exp[0]*100:.1f}%)', fontsize=12)
ax.set_ylabel(f'PC2 ({var_exp[1]*100:.1f}%)', fontsize=12)
ax.set_title('PCA – ESM-2 Embeddings (1,280D → 2D)', fontsize=13,
             fontweight='bold')
ax.legend(markerscale=3, fontsize=11)

# ── Right: explained variance ───────────────────────────────────────────────
ax2 = axes[1]
ax2.plot(np.arange(1, len(cum_var) + 1), cum_var * 100,
         color='steelblue', linewidth=2)
ax2.axhline(90, linestyle='--', color='grey', linewidth=1)
ax2.fill_between(np.arange(1, len(cum_var) + 1), cum_var * 100,
                 alpha=0.15, color='steelblue')
ax2.set_xlabel('Number of Principal Components', fontsize=12)
ax2.set_ylabel('Cumulative Explained Variance (%)', fontsize=12)
ax2.set_title('PCA – Explained Variance', fontsize=13, fontweight='bold')
ax2.set_xlim(1, len(cum_var))

plt.tight_layout()
pca_fig_path = os.path.join(FIG_DIR, '03_03_pca_projection.png')
plt.savefig(pca_fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {pca_fig_path}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── L2 norms ────────────────────────────────────────────────────────────────
ax = axes[0]
for label, colour in colours.items():
    mask  = labels_array == label
    norms = np.linalg.norm(embedding_matrix[mask], axis=1)
    ax.hist(norms, bins=50, color=colour, alpha=0.7, label=label,
            edgecolor='white')
ax.set_xlabel('Embedding L2 norm', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Embedding Magnitude Distribution', fontsize=13,
             fontweight='bold')
ax.legend(fontsize=11)

# ── Mean embedding values ────────────────────────────────────────────────────
ax2 = axes[1]
for label, colour in colours.items():
    mask = labels_array == label
    means = embedding_matrix[mask].mean(axis=0)
    ax2.plot(means, color=colour, alpha=0.8, linewidth=0.8, label=label)
ax2.set_xlabel('Embedding dimension', fontsize=12)
ax2.set_ylabel('Mean value', fontsize=12)
ax2.set_title('Mean Embedding Values per Dimension', fontsize=13,
              fontweight='bold')
ax2.legend(fontsize=11)

plt.tight_layout()
dist_fig_path = os.path.join(FIG_DIR, '03_02_embedding_distribution.png')
plt.savefig(dist_fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {dist_fig_path}')

## Section 9 – Clustering Quality Metrics

In [ ]:
def centroid(emb_subset):
    return emb_subset.mean(axis=0)

c_lassa = centroid(embedding_matrix[lassa_mask])
c_ebola = centroid(embedding_matrix[ebola_mask])

inter_distance = float(np.linalg.norm(c_lassa - c_ebola))

intra_lassa = float(np.mean(
    np.linalg.norm(embedding_matrix[lassa_mask] - c_lassa, axis=1)))
intra_ebola = float(np.mean(
    np.linalg.norm(embedding_matrix[ebola_mask] - c_ebola, axis=1)))

separation_ratio = inter_distance / max(intra_lassa, intra_ebola)

print('── Clustering metrics ──────────────────────────────')
print(f'Inter-centroid distance  : {inter_distance:.4f}')
print(f'Intra-class spread Lassa : {intra_lassa:.4f}')
print(f'Intra-class spread Ebola : {intra_ebola:.4f}')
print(f'Separation ratio         : {separation_ratio:.4f}',
      ' (>1.0 = well-separated)')

In [ ]:
# Sample up to 50 representative sequences per class for the heatmap
N_SAMPLE = 50
rng = np.random.default_rng(SEED)

lassa_idx = np.where(lassa_mask)[0]
ebola_idx = np.where(ebola_mask)[0]
s_lassa = rng.choice(lassa_idx, size=min(N_SAMPLE, len(lassa_idx)),
                     replace=False)
s_ebola = rng.choice(ebola_idx, size=min(N_SAMPLE, len(ebola_idx)),
                     replace=False)
s_all   = np.concatenate([s_lassa, s_ebola])
sub_emb = embedding_matrix[s_all]
sub_lbl = labels_array[s_all]

# Cosine distance matrix
from sklearn.metrics.pairwise import cosine_distances
D = cosine_distances(sub_emb)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(D, cmap='RdYlBu_r', aspect='auto', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label='Cosine distance')

n_l = len(s_lassa)
ax.axhline(n_l - 0.5, color='black', linewidth=1.5)
ax.axvline(n_l - 0.5, color='black', linewidth=1.5)
ax.text(n_l / 2, -2, 'Lassa', ha='center', va='bottom', fontsize=10,
        fontweight='bold', color='#2196F3')
ax.text(n_l + len(s_ebola) / 2, -2, 'Ebola', ha='center', va='bottom',
        fontsize=10, fontweight='bold', color='#F44336')
ax.set_title('Pairwise Cosine Distance Matrix\n(Sampled sequences)',
             fontsize=13, fontweight='bold')
ax.set_xticks([])
ax.set_yticks([])
plt.tight_layout()

heat_path = os.path.join(FIG_DIR, '03_04_distance_matrices.png')
plt.savefig(heat_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {heat_path}')

## Section 10 – Summary Report

In [ ]:
summary = {
    'model': ESM_MODEL_NAME,
    'embedding_dimension': EMBED_DIM,
    'total_sequences': int(len(all_records)),
    'lassa_count': int(lassa_mask.sum()),
    'ebola_count': int(ebola_mask.sum()),
    'failed_sequences': len(failed_ids),
    'embedding_matrix_shape': list(embedding_matrix.shape),
    'pca': {
        'pc1_variance_pct': float(round(var_exp[0] * 100, 2)),
        'pc2_variance_pct': float(round(var_exp[1] * 100, 2)),
        'components_for_90pct': int(n90),
    },
    'clustering': {
        'inter_centroid_distance': float(round(inter_distance, 4)),
        'intra_spread_lassa': float(round(intra_lassa, 4)),
        'intra_spread_ebola': float(round(intra_ebola, 4)),
        'separation_ratio': float(round(separation_ratio, 4)),
    },
    'output_files': [
        npz_path, lassa_npy, ebola_npy, meta_path,
        pca_fig_path, dist_fig_path, heat_path,
    ],
}

summary_path = os.path.join(EMBED_DIR, 'summary.json')
with open(summary_path, 'w') as fh:
    json.dump(summary, fh, indent=2)

print('── Summary ─────────────────────────────────────────────────────')
print(f'  Model                 : {summary["model"]}')
print(f'  Embedding dimension   : {summary["embedding_dimension"]}')
print(f'  Total sequences       : {summary["total_sequences"]}')
print(f'  Lassa / Ebola         : {summary["lassa_count"]} / {summary["ebola_count"]}')
print(f'  Separation ratio      : {summary["clustering"]["separation_ratio"]}')
print(f'  Summary saved         : {summary_path}')
print('────────────────────────────────────────────────────────────────')
print('Notebook complete. ✓')